# Pseudo-Top Uncertainty Feature Screen

This notebook turns train-only formation surface columns into fold-safe latent stratigraphic features. Direct formation columns are used only as local labels and oracle diagnostics; legal candidates see only row geometry, known-zone summaries, typewell alignment descriptors, and fold-trained pseudo-top predictions.

## 1. Purpose And Leakage Rules

Formation columns (`ANCC`, `ASTNU`, `ASTNL`, `EGFDU`, `EGFDL`, `BUDA`) are row-level formation-surface elevations in local train files. Hidden test files do not expose them, so this notebook learns proxy top elevations from hidden-test-safe predictors inside each validation fold.

The downstream convention is:

`pseudo_signed_offset_<formation> = Z - pseudo_top_elev_<formation>`

This keeps the top-elevation label explicit and lets downstream models use signed/absolute offsets, nearest-top margin, and ensemble uncertainty without direct formation leakage.

## 2. Setup

In [1]:
"Configure imports, paths, and deterministic settings."

from pathlib import Path
import json
import shutil
import time

import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import HuberRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor

pd.set_option("display.max_columns", 180)
pd.set_option("display.width", 220)

ROOT = Path.cwd()
if not (ROOT / "context").exists():
    ROOT = next(parent for parent in ROOT.parents if (parent / "context").exists())

RUN_DATE = "2026-06-30"
RUN_TAG = "pseudo_top_uncertainty"
RUN_ID = f"{RUN_DATE}__{RUN_TAG}"
SEED = 20260630
FORMATION_COLS = ["ANCC", "ASTNU", "ASTNL", "EGFDU", "EGFDL", "BUDA"]
PRIMARY_FOLD = "strict_hybrid_fold"
STRESS_FOLD = "analog_distance_fold"
MAX_TARGET_ROWS_PER_WELL = 120
ENSEMBLE_SEEDS = [20260630, 20260631]

DATA_DIR = ROOT / "data" / "raw"
TRAIN_DIR = DATA_DIR / "train"
TABLE_DIR = ROOT / "artifacts" / "tables"
REPORT_DIR = ROOT / "artifacts" / "reports"
RUN_DIR = ROOT / "artifacts" / "logs" / "runs" / RUN_ID
for directory in [TABLE_DIR, REPORT_DIR, RUN_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "run_id": RUN_ID,
    "seed": SEED,
    "formation_cols": FORMATION_COLS,
    "primary_fold": PRIMARY_FOLD,
    "stress_fold": STRESS_FOLD,
    "max_target_rows_per_well": MAX_TARGET_ROWS_PER_WELL,
    "ensemble_seeds": ENSEMBLE_SEEDS,
    "current_public_best_rmse": 14.472,
    "best_xgb_known_state_local_rmse": 13.4334,
    "best_xgb_known_state_stress_rmse": 13.7533,
    "promotion_margin": 0.05,
}
with (RUN_DIR / "config.json").open("w", encoding="utf-8") as handle:
    json.dump(CONFIG, handle, indent=2)

print(ROOT)
print(RUN_DIR)

C:\Users\sardo\OneDrive\Desktop\Classes\Projects\Wellbore-tvt-Modeling
C:\Users\sardo\OneDrive\Desktop\Classes\Projects\Wellbore-tvt-Modeling\artifacts\logs\runs\2026-06-30__pseudo_top_uncertainty


## 3. Data Loading And Label Audit

In [2]:
"Build sampled target-zone rows with legal predictors and direct top-elevation labels."

folds = pd.read_csv(TABLE_DIR / "validation__strict_cv_fold_assignments__2026-06-26.csv")
descriptors = pd.read_csv(TABLE_DIR / "model__typewell_alignment_feature_descriptors__2026-06-23.csv")
well_meta = folds.merge(descriptors, on="wellname", how="left")
well_meta = well_meta.drop_duplicates("wellname").reset_index(drop=True)

DESCRIPTOR_COLS = [
    "known_rows", "known_tvt_span", "known_md_span", "known_gr_mean_last300", "known_gr_std_last300", "known_gr_slope_last300",
    "xy_span_known", "z_drop_known", "typewell_corr_last300", "typewell_mae_last300", "typewell_best_lag_corr",
    "typewell_best_lag_tvt", "typewell_best_lag_mae", "selected_analog_distance", "extrapolation_stress_score",
    "analog_stress_score", "combined_stress_score", "descriptor_cluster",
]


def safe_divide(num, den):
    return num / (den + 1e-9)


def load_typewell(directory, wellname):
    path = Path(directory) / f"{wellname}__typewell.csv"
    if not path.exists():
        return pd.DataFrame(columns=["TVT", "GR", "Geology"])
    return pd.read_csv(path)


def interp_typewell_gr(typewell, tvt_values):
    if typewell.empty or not {"TVT", "GR"}.issubset(typewell.columns):
        return np.full(len(tvt_values), np.nan)
    clean = typewell[["TVT", "GR"]].dropna().sort_values("TVT")
    if len(clean) < 5:
        return np.full(len(tvt_values), np.nan)
    return np.interp(np.asarray(tvt_values, dtype=float), clean["TVT"].to_numpy(), clean["GR"].to_numpy(), left=np.nan, right=np.nan)


def build_target_rows(csv_path, meta_row):
    df = pd.read_csv(csv_path).sort_values("MD").reset_index(drop=True)
    known_mask = df["TVT_input"].notna()
    if known_mask.sum() == 0 or (~known_mask).sum() == 0:
        return pd.DataFrame()
    last_known_idx = int(np.flatnonzero(known_mask.to_numpy())[-1])
    known = df.loc[:last_known_idx].copy()
    target = df.loc[last_known_idx + 1:].copy()
    target = target.loc[target["TVT"].notna()].copy()
    if target.empty:
        return pd.DataFrame()
    if len(target) > MAX_TARGET_ROWS_PER_WELL:
        sample_idx = np.linspace(0, len(target) - 1, MAX_TARGET_ROWS_PER_WELL).round().astype(int)
        target = target.iloc[np.unique(sample_idx)].copy()

    typewell = load_typewell(csv_path.parent, meta_row["wellname"])
    last = known.iloc[-1]
    first = df.iloc[0]
    known_gr = known["GR"].astype(float).clip(0, 300)
    known_gr_finite = known_gr.dropna()
    gr_fill = float(known_gr.tail(300).median()) if known_gr.notna().any() else 0.0
    target_gr = target["GR"].astype(float).clip(0, 300).fillna(gr_fill)

    known_md_span_state = float(known["MD"].iloc[-1] - known["MD"].iloc[0]) if len(known) > 1 else 0.0
    known_tvt_delta_state = float(known["TVT_input"].iloc[-1] - known["TVT_input"].iloc[0]) if len(known) > 1 else 0.0
    known_tvt_slope_state = known_tvt_delta_state / max(abs(known_md_span_state), 1.0)
    known_xy_span_state = float(np.hypot(known["X"].iloc[-1] - known["X"].iloc[0], known["Y"].iloc[-1] - known["Y"].iloc[0])) if len(known) > 1 else 0.0
    known_z_drop_state = float(known["Z"].iloc[-1] - known["Z"].iloc[0]) if len(known) > 1 else 0.0
    known_gr_coverage_state = float(known_gr.notna().mean()) if len(known_gr) else 0.0
    known_gr_last_state = float(known_gr_finite.iloc[-1]) if len(known_gr_finite) else gr_fill
    known_gr_iqr_state = float(known_gr_finite.quantile(0.75) - known_gr_finite.quantile(0.25)) if len(known_gr_finite) >= 4 else 0.0

    target = target.reset_index(drop=False).rename(columns={"index": "row_idx_original"})
    out = pd.DataFrame({
        "wellname": meta_row["wellname"],
        "row_idx_original": target["row_idx_original"].astype(float),
        "MD": target["MD"].astype(float),
        "X": target["X"].astype(float),
        "Y": target["Y"].astype(float),
        "Z": target["Z"].astype(float),
        "GR": target_gr.to_numpy(),
        "TVT": target["TVT"].astype(float),
        "last_known_tvt_feature": float(last["TVT"]),
        "last_known_md": float(last["MD"]),
        "last_known_x": float(last["X"]),
        "last_known_y": float(last["Y"]),
        "last_known_z": float(last["Z"]),
        "first_md": float(first["MD"]),
        "state_known_tvt_last": float(last["TVT_input"] if pd.notna(last.get("TVT_input", np.nan)) else last["TVT"]),
        "state_known_tvt_slope": known_tvt_slope_state,
        "state_known_gr_coverage": known_gr_coverage_state,
        "state_known_gr_last": known_gr_last_state,
        "state_known_gr_iqr": known_gr_iqr_state,
        "state_known_xy_span": known_xy_span_state,
        "state_known_z_drop": known_z_drop_state,
        "state_known_md_span": known_md_span_state,
    })
    out["target_delta"] = out["TVT"] - out["last_known_tvt_feature"]
    out["md_from_last_known"] = out["MD"] - out["last_known_md"]
    out["md_from_start"] = out["MD"] - out["first_md"]
    out["eval_progress"] = safe_divide(np.arange(1, len(out) + 1), len(out))
    out["eval_progress_sq"] = out["eval_progress"] ** 2
    out["dx_from_last"] = out["X"] - out["last_known_x"]
    out["dy_from_last"] = out["Y"] - out["last_known_y"]
    out["dz_from_last"] = out["Z"] - out["last_known_z"]
    out["horizontal_from_last"] = np.sqrt(out["dx_from_last"] ** 2 + out["dy_from_last"] ** 2)
    out["distance_from_last"] = np.sqrt(out["horizontal_from_last"] ** 2 + out["dz_from_last"] ** 2)
    out["distance_log1p"] = np.log1p(out["distance_from_last"].clip(lower=0))
    out["tvt_linear_proxy"] = out["last_known_tvt_feature"] + out["md_from_last_known"]
    best_lag = float(meta_row.get("typewell_best_lag_tvt", 0.0) if pd.notna(meta_row.get("typewell_best_lag_tvt", np.nan)) else 0.0)
    out["typewell_gr_linear_tvt"] = interp_typewell_gr(typewell, out["tvt_linear_proxy"].to_numpy())
    out["typewell_gr_lag_best"] = interp_typewell_gr(typewell, out["tvt_linear_proxy"].to_numpy() + best_lag)
    out["gr_minus_typewell_linear"] = out["GR"] - out["typewell_gr_linear_tvt"]
    out["gr_minus_typewell_bestlag"] = out["GR"] - out["typewell_gr_lag_best"]

    for formation in FORMATION_COLS:
        out[f"formation_top_{formation}"] = target[formation].astype(float).to_numpy()
        out[f"label_signed_offset_{formation}"] = out["Z"] - out[f"formation_top_{formation}"]
        out[f"label_abs_offset_{formation}"] = np.abs(out[f"label_signed_offset_{formation}"])

    abs_label_cols = [f"label_abs_offset_{formation}" for formation in FORMATION_COLS]
    out["label_nearest_abs_offset"] = out[abs_label_cols].min(axis=1)
    out["label_nearest_formation"] = out[abs_label_cols].idxmin(axis=1).str.replace("label_abs_offset_", "", regex=False)
    out["label_mean_abs_offset"] = out[abs_label_cols].mean(axis=1)

    for col in [PRIMARY_FOLD, STRESS_FOLD, "descriptor_cluster_fold", "extrapolation_fold"]:
        out[col] = int(meta_row[col])
    for col in DESCRIPTOR_COLS:
        out[col] = float(meta_row[col]) if pd.notna(meta_row[col]) else np.nan
    return out


frames = []
for meta_row in well_meta.to_dict(orient="records"):
    csv_path = TRAIN_DIR / f"{meta_row['wellname']}__horizontal_well.csv"
    if csv_path.exists():
        frame = build_target_rows(csv_path, meta_row)
        if not frame.empty:
            frames.append(frame)

data = pd.concat(frames, ignore_index=True)

SAFE_ROW_COLS = [
    "row_idx_original", "MD", "X", "Y", "Z", "GR", "last_known_tvt_feature", "last_known_md",
    "md_from_last_known", "md_from_start", "eval_progress", "eval_progress_sq",
    "dx_from_last", "dy_from_last", "dz_from_last", "horizontal_from_last", "distance_from_last", "distance_log1p",
]
SAFE_FEATURE_COLS = [col for col in [*SAFE_ROW_COLS, *DESCRIPTOR_COLS] if col in data.columns]
TYPEWELL_FEATURE_COLS = [
    "tvt_linear_proxy", "typewell_gr_linear_tvt", "typewell_gr_lag_best", "gr_minus_typewell_linear", "gr_minus_typewell_bestlag",
]
TYPEWELL_FEATURE_COLS = [col for col in TYPEWELL_FEATURE_COLS if col in data.columns]
STATE_FEATURE_COLS = [
    "state_known_tvt_last", "state_known_tvt_slope", "state_known_gr_coverage", "state_known_gr_last",
    "state_known_gr_iqr", "state_known_xy_span", "state_known_z_drop", "state_known_md_span",
]
DIRECT_FORMATION_FEATURE_COLS = [
    *[f"label_signed_offset_{formation}" for formation in FORMATION_COLS],
    *[f"label_abs_offset_{formation}" for formation in FORMATION_COLS],
    "label_nearest_abs_offset", "label_mean_abs_offset",
]

top_label_summary = []
for formation in FORMATION_COLS:
    values = data[f"formation_top_{formation}"].astype(float)
    offsets = data[f"label_signed_offset_{formation}"].astype(float)
    top_label_summary.append({
        "formation": formation,
        "top_mean": float(values.mean()),
        "top_std": float(values.std()),
        "top_min": float(values.min()),
        "top_max": float(values.max()),
        "signed_offset_mean": float(offsets.mean()),
        "signed_offset_std": float(offsets.std()),
        "abs_offset_mean": float(np.abs(offsets).mean()),
    })
top_label_summary = pd.DataFrame(top_label_summary)
label_counts = data["label_nearest_formation"].value_counts().rename_axis("formation").reset_index(name="nearest_label_rows")
top_label_summary = top_label_summary.merge(label_counts, on="formation", how="left").fillna({"nearest_label_rows": 0})

label_path = TABLE_DIR / "model__pseudo_top_uncertainty_top_label_summary__2026-06-30.csv"
top_label_summary.to_csv(label_path, index=False)
shutil.copy2(label_path, RUN_DIR / label_path.name)

feature_preview_path = TABLE_DIR / "model__pseudo_top_uncertainty_feature_preview__2026-06-30.csv"
data.head(500).to_csv(feature_preview_path, index=False)
shutil.copy2(feature_preview_path, RUN_DIR / feature_preview_path.name)

print(data.shape)
print("safe_features", len(SAFE_FEATURE_COLS))
print(top_label_summary.to_string(index=False))

(92760, 81)
safe_features 36
formation     top_mean    top_std   top_min  top_max  signed_offset_mean  signed_offset_std  abs_offset_mean  nearest_label_rows
     ANCC -9008.099049 633.914859 -10647.31 -7301.76         -416.172460          55.411650       416.172460                 0.0
    ASTNU -9190.780447 639.418693 -10852.79 -7509.20         -244.653750          63.560087       244.653750                 0.0
    ASTNL -9255.299324 624.857096 -10865.58 -7614.43         -180.134874          51.475069       180.135674              2455.0
    EGFDU -9349.765034 619.827567 -10887.58 -7704.03          -85.669164          43.889718        91.053770              3919.0
    EGFDL -9391.937722 619.776843 -10932.35 -7731.61          -43.798726          44.572447        55.375403             50858.0
     BUDA -9516.381964 633.862561 -11104.73 -7835.57           80.947766          45.807212        80.953400             35528.0


## 4. Fold-Safe Pseudo-Top Feature Builders

In [3]:
"Train fold-safe top-elevation ensembles and derive uncertainty, margin, and nearest-top features."

PSEUDO_TOP_PREDICTOR_COLS = [
    *SAFE_FEATURE_COLS,
    *TYPEWELL_FEATURE_COLS,
    *STATE_FEATURE_COLS,
]
PSEUDO_TOP_PREDICTOR_COLS = list(dict.fromkeys([col for col in PSEUDO_TOP_PREDICTOR_COLS if col in data.columns]))

pseudo_params = {
    "objective": "reg:squarederror",
    "n_estimators": 260,
    "learning_rate": 0.040,
    "max_depth": 3,
    "min_child_weight": 90,
    "subsample": 0.85,
    "colsample_bytree": 0.85,
    "reg_alpha": 0.20,
    "reg_lambda": 10.0,
    "tree_method": "hist",
    "n_jobs": 4,
    "verbosity": 0,
}


def rmse(y_true, y_pred):
    return float(mean_squared_error(y_true, y_pred) ** 0.5)


def softmax_neg_abs(abs_matrix, scale=50.0):
    logits = -abs_matrix / scale
    logits = logits - np.nanmax(logits, axis=1, keepdims=True)
    exp_logits = np.exp(logits)
    return exp_logits / np.clip(exp_logits.sum(axis=1, keepdims=True), 1e-12, None)


def add_pseudo_features_for_fold_column(frame, fold_col):
    out = frame.copy()
    proxy_quality_rows = []
    for formation in FORMATION_COLS:
        out[f"pseudo_top_elev_{formation}__{fold_col}"] = np.nan
        out[f"pseudo_top_std_{formation}__{fold_col}"] = np.nan

    X_all = out[PSEUDO_TOP_PREDICTOR_COLS].replace([np.inf, -np.inf], np.nan)
    for fold in sorted(out[fold_col].dropna().unique()):
        valid_mask = out[fold_col].eq(fold)
        train_mask = ~valid_mask
        X_train = X_all.loc[train_mask]
        X_valid = X_all.loc[valid_mask]
        for formation in FORMATION_COLS:
            target_col = f"formation_top_{formation}"
            label_train_mask = train_mask & out[target_col].notna()
            label_valid_mask = valid_mask & out[target_col].notna()
            seed_predictions = []
            if label_train_mask.sum() >= 20:
                X_label_train = X_all.loc[label_train_mask]
                y_train = out.loc[label_train_mask, target_col].astype(float)
                for seed in ENSEMBLE_SEEDS:
                    model = XGBRegressor(**pseudo_params, random_state=seed)
                    model.fit(X_label_train, y_train, verbose=False)
                    seed_predictions.append(model.predict(X_valid))
                pred_matrix = np.column_stack(seed_predictions)
                pred_mean = pred_matrix.mean(axis=1)
                pred_std = pred_matrix.std(axis=1)
            else:
                pred_mean = np.full(valid_mask.sum(), float(out[target_col].median()))
                pred_std = np.zeros(valid_mask.sum())
            out.loc[valid_mask, f"pseudo_top_elev_{formation}__{fold_col}"] = pred_mean
            out.loc[valid_mask, f"pseudo_top_std_{formation}__{fold_col}"] = pred_std
            if label_valid_mask.sum() > 0:
                valid_index = out.loc[valid_mask].index
                pred_series = pd.Series(pred_mean, index=valid_index).loc[label_valid_mask]
                y_valid = out.loc[label_valid_mask, target_col].astype(float)
                true_offset = out.loc[label_valid_mask, "Z"].to_numpy(float) - y_valid.to_numpy(float)
                pred_offset = out.loc[label_valid_mask, "Z"].to_numpy(float) - pred_series.to_numpy(float)
                corr = float(np.corrcoef(y_valid.to_numpy(float), pred_series.to_numpy(float))[0, 1]) if len(y_valid) > 2 else np.nan
                top_rmse = rmse(y_valid, pred_series)
                top_mae = float(mean_absolute_error(y_valid, pred_series))
                signed_offset_rmse = rmse(true_offset, pred_offset)
            else:
                corr = np.nan
                top_rmse = np.nan
                top_mae = np.nan
                signed_offset_rmse = np.nan
            proxy_quality_rows.append({
                "fold_column": fold_col,
                "fold": int(fold),
                "formation": formation,
                "n_train": int(label_train_mask.sum()),
                "n_valid": int(label_valid_mask.sum()),
                "top_rmse": top_rmse,
                "top_mae": top_mae,
                "top_corr": corr,
                "signed_offset_rmse": signed_offset_rmse,
                "mean_ensemble_std": float(np.mean(pred_std)),
            })

    for formation in FORMATION_COLS:
        top_col = f"pseudo_top_elev_{formation}__{fold_col}"
        std_col = f"pseudo_top_std_{formation}__{fold_col}"
        out[top_col] = out[top_col].fillna(float(out[f"formation_top_{formation}"].median()))
        out[std_col] = out[std_col].fillna(float(out[std_col].median()) if out[std_col].notna().any() else 0.0)
        out[f"pseudo_signed_offset_{formation}__{fold_col}"] = out["Z"] - out[top_col]
        out[f"pseudo_abs_offset_{formation}__{fold_col}"] = np.abs(out[f"pseudo_signed_offset_{formation}__{fold_col}"])

    abs_cols = [f"pseudo_abs_offset_{formation}__{fold_col}" for formation in FORMATION_COLS]
    std_cols = [f"pseudo_top_std_{formation}__{fold_col}" for formation in FORMATION_COLS]
    abs_matrix = out[abs_cols].to_numpy(float)
    probs = softmax_neg_abs(abs_matrix)
    sorted_abs = np.sort(abs_matrix, axis=1)
    sorted_probs = np.sort(probs, axis=1)[:, ::-1]
    nearest_idx = np.argmin(abs_matrix, axis=1)

    out[f"pseudo_nearest_abs_offset__{fold_col}"] = sorted_abs[:, 0]
    out[f"pseudo_second_abs_offset__{fold_col}"] = sorted_abs[:, 1]
    out[f"pseudo_top_margin__{fold_col}"] = out[f"pseudo_second_abs_offset__{fold_col}"] - out[f"pseudo_nearest_abs_offset__{fold_col}"]
    out[f"pseudo_top_confidence__{fold_col}"] = 1.0 / (1.0 + np.exp(-out[f"pseudo_top_margin__{fold_col}"].clip(-100, 100) / 25.0))
    out[f"pseudo_near_boundary_25__{fold_col}"] = (out[f"pseudo_nearest_abs_offset__{fold_col}"] <= 25.0).astype(float)
    out[f"pseudo_uncertain_boundary__{fold_col}"] = ((out[f"pseudo_nearest_abs_offset__{fold_col}"] <= 50.0) & (out[f"pseudo_top_margin__{fold_col}"] <= 25.0)).astype(float)
    out[f"pseudo_mean_abs_offset__{fold_col}"] = abs_matrix.mean(axis=1)
    out[f"pseudo_min_std__{fold_col}"] = out[std_cols].min(axis=1)
    out[f"pseudo_mean_std__{fold_col}"] = out[std_cols].mean(axis=1)
    out[f"pseudo_top1_prob__{fold_col}"] = sorted_probs[:, 0]
    out[f"pseudo_top2_prob__{fold_col}"] = sorted_probs[:, 1]
    out[f"pseudo_top_entropy__{fold_col}"] = -(probs * np.log(probs + 1e-12)).sum(axis=1)
    out[f"pseudo_nearest_formation__{fold_col}"] = [FORMATION_COLS[int(idx)] for idx in nearest_idx]
    out[f"pseudo_nearest_matches_label__{fold_col}"] = (out[f"pseudo_nearest_formation__{fold_col}"] == out["label_nearest_formation"]).astype(float)

    return out, pd.DataFrame(proxy_quality_rows)


pseudo_feature_cols_by_fold = {}
pseudo_uncertainty_cols_by_fold = {}
quality_parts = []
for fold_col in [PRIMARY_FOLD, STRESS_FOLD]:
    data, quality = add_pseudo_features_for_fold_column(data, fold_col)
    quality_parts.append(quality)
    pseudo_feature_cols_by_fold[fold_col] = [
        *[f"pseudo_signed_offset_{formation}__{fold_col}" for formation in FORMATION_COLS],
        *[f"pseudo_abs_offset_{formation}__{fold_col}" for formation in FORMATION_COLS],
        f"pseudo_nearest_abs_offset__{fold_col}",
        f"pseudo_second_abs_offset__{fold_col}",
        f"pseudo_mean_abs_offset__{fold_col}",
    ]
    pseudo_uncertainty_cols_by_fold[fold_col] = [
        *[f"pseudo_top_std_{formation}__{fold_col}" for formation in FORMATION_COLS],
        f"pseudo_top_margin__{fold_col}",
        f"pseudo_top_confidence__{fold_col}",
        f"pseudo_near_boundary_25__{fold_col}",
        f"pseudo_uncertain_boundary__{fold_col}",
        f"pseudo_min_std__{fold_col}",
        f"pseudo_mean_std__{fold_col}",
        f"pseudo_top1_prob__{fold_col}",
        f"pseudo_top2_prob__{fold_col}",
        f"pseudo_top_entropy__{fold_col}",
    ]

proxy_quality = pd.concat(quality_parts, ignore_index=True)
nearest_quality = []
for fold_col in [PRIMARY_FOLD, STRESS_FOLD]:
    nearest_quality.append({
        "fold_column": fold_col,
        "fold": "all",
        "formation": "nearest_classifier",
        "n_train": np.nan,
        "n_valid": int(len(data)),
        "top_rmse": np.nan,
        "top_mae": np.nan,
        "top_corr": np.nan,
        "signed_offset_rmse": np.nan,
        "mean_ensemble_std": float(data[f"pseudo_mean_std__{fold_col}"].mean()),
        "nearest_accuracy": float(data[f"pseudo_nearest_matches_label__{fold_col}"].mean()),
    })
proxy_quality["nearest_accuracy"] = np.nan
proxy_quality = pd.concat([proxy_quality, pd.DataFrame(nearest_quality)], ignore_index=True)

proxy_quality_path = TABLE_DIR / "model__pseudo_top_uncertainty_proxy_quality__2026-06-30.csv"
proxy_quality.to_csv(proxy_quality_path, index=False)
shutil.copy2(proxy_quality_path, RUN_DIR / proxy_quality_path.name)
proxy_quality.groupby(["fold_column", "formation"], dropna=False).agg(top_rmse=("top_rmse", "mean"), top_mae=("top_mae", "mean"), mean_ensemble_std=("mean_ensemble_std", "mean"), nearest_accuracy=("nearest_accuracy", "mean")).reset_index()

,fold_column,formation,top_rmse,top_mae,mean_ensemble_std,nearest_accuracy
0,analog_distance_fold,ANCC,36.115667,27.307085,2.252600,NaN
1,analog_distance_fold,ASTNL,36.141518,26.902796,2.586729,NaN
2,analog_distance_fold,ASTNU,37.748549,28.102448,2.489752,NaN
3,analog_distance_fold,BUDA,34.471779,25.013579,2.370127,NaN
4,analog_distance_fold,EGFDL,34.509167,25.367227,2.504926,NaN
5,analog_distance_fold,EGFDU,34.209112,25.063224,2.354379,NaN
6,analog_distance_fold,nearest_classifier,NaN,NaN,2.426611,0.582277
7,strict_hybrid_fold,ANCC,35.208638,26.607742,2.021169,NaN
8,strict_hybrid_fold,ASTNL,35.230138,26.536828,2.328327,NaN
9,strict_hybrid_fold,ASTNU,36.706530,27.494402,2.655390,NaN


## 5. Downstream Model Consumers

In [4]:
"Define feature sets, consumers, and fold-safe evaluation helpers."

xgb_base_params = {
    "tree_method": "hist",
    "random_state": SEED,
    "n_jobs": 4,
    "verbosity": 0,
    "eval_metric": "rmse",
}
xgb_candidates = {
    "xgb_pseudohuber_known_state_recipe": {
        "objective": "reg:pseudohubererror",
        "n_estimators": 700,
        "learning_rate": 0.030,
        "max_depth": 4,
        "min_child_weight": 100,
        "subsample": 0.85,
        "colsample_bytree": 0.85,
        "reg_alpha": 0.15,
        "reg_lambda": 7.0,
        "gamma": 0.05,
    },
    "xgb_square_regularized": {
        "objective": "reg:squarederror",
        "n_estimators": 550,
        "learning_rate": 0.035,
        "max_depth": 5,
        "min_child_weight": 90,
        "subsample": 0.85,
        "colsample_bytree": 0.85,
        "reg_alpha": 0.20,
        "reg_lambda": 8.0,
        "gamma": 0.10,
    },
}


def feature_sets_for_fold(fold_col):
    return {
        "safe_base": SAFE_FEATURE_COLS,
        "safe_typewell": [*SAFE_FEATURE_COLS, *TYPEWELL_FEATURE_COLS],
        "safe_typewell_state": [*SAFE_FEATURE_COLS, *TYPEWELL_FEATURE_COLS, *STATE_FEATURE_COLS],
        "safe_pseudo_top": [*SAFE_FEATURE_COLS, *TYPEWELL_FEATURE_COLS, *pseudo_feature_cols_by_fold[fold_col]],
        "safe_pseudo_top_uncertainty": [*SAFE_FEATURE_COLS, *TYPEWELL_FEATURE_COLS, *pseudo_feature_cols_by_fold[fold_col], *pseudo_uncertainty_cols_by_fold[fold_col]],
        "oracle_direct_formations": [*SAFE_FEATURE_COLS, *TYPEWELL_FEATURE_COLS, *STATE_FEATURE_COLS, *DIRECT_FORMATION_FEATURE_COLS],
    }


def metric_row(actual, pred, baseline):
    return {
        "rmse": rmse(actual, pred),
        "mae": float(mean_absolute_error(actual, pred)),
        "r2": float(r2_score(actual, pred)),
        "last_known_hold_rmse": rmse(actual, baseline),
    }


def fit_huber_predict(X_train, y_train, X_valid):
    model = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), HuberRegressor(alpha=0.0001, epsilon=1.35, max_iter=900))
    model.fit(X_train, y_train)
    return model.predict(X_valid), model


def evaluate_fold_column(fold_col, include_oracle=True):
    feature_sets = feature_sets_for_fold(fold_col)
    if not include_oracle:
        feature_sets = {key: value for key, value in feature_sets.items() if key != "oracle_direct_formations"}
    fold_rows = []
    pred_parts = []
    importance_parts = []
    y_delta = data["target_delta"].astype(float)
    y_tvt = data["TVT"].astype(float)
    baseline = data["last_known_tvt_feature"].astype(float)

    for feature_set_name, raw_cols in feature_sets.items():
        feature_cols = list(dict.fromkeys([col for col in raw_cols if col in data.columns]))
        X_all = data[feature_cols].replace([np.inf, -np.inf], np.nan)
        consumers = [("huber_delta", None)]
        if feature_set_name != "oracle_direct_formations":
            consumers.extend((name, params) for name, params in xgb_candidates.items())
        else:
            consumers.append(("xgb_pseudohuber_known_state_recipe", xgb_candidates["xgb_pseudohuber_known_state_recipe"]))

        for consumer_name, params in consumers:
            label = f"{consumer_name}__{feature_set_name}"
            for fold in sorted(data[fold_col].dropna().unique()):
                valid_mask = data[fold_col].eq(fold)
                train_mask = ~valid_mask
                if consumer_name == "huber_delta":
                    pred_delta, model = fit_huber_predict(X_all.loc[train_mask], y_delta.loc[train_mask], X_all.loc[valid_mask])
                    best_iteration = np.nan
                else:
                    model = XGBRegressor(**xgb_base_params, **params)
                    model.fit(X_all.loc[train_mask], y_delta.loc[train_mask], verbose=False)
                    pred_delta = model.predict(X_all.loc[valid_mask])
                    best_iteration = int(params["n_estimators"])
                    importance_parts.append(pd.DataFrame({
                        "fold_column": fold_col,
                        "fold": int(fold),
                        "label": label,
                        "consumer": consumer_name,
                        "feature_set": feature_set_name,
                        "feature": feature_cols,
                        "importance": model.feature_importances_,
                    }))

                pred_tvt = baseline.loc[valid_mask].to_numpy() + pred_delta
                actual = y_tvt.loc[valid_mask].to_numpy()
                base = baseline.loc[valid_mask].to_numpy()
                metrics = metric_row(actual, pred_tvt, base)
                fold_rows.append({
                    "run_id": RUN_ID,
                    "fold_column": fold_col,
                    "fold": int(fold),
                    "label": label,
                    "consumer": consumer_name,
                    "feature_set": feature_set_name,
                    "n_train_rows": int(train_mask.sum()),
                    "n_valid_rows": int(valid_mask.sum()),
                    "n_features": int(len(feature_cols)),
                    "best_iteration": best_iteration,
                    **metrics,
                })
                tmp = data.loc[valid_mask, ["wellname", PRIMARY_FOLD, STRESS_FOLD, "eval_progress", "selected_analog_distance", "combined_stress_score", f"pseudo_nearest_abs_offset__{fold_col}", f"pseudo_top_margin__{fold_col}", f"pseudo_mean_std__{fold_col}"]].copy()
                tmp["fold_column"] = fold_col
                tmp["fold"] = int(fold)
                tmp["label"] = label
                tmp["consumer"] = consumer_name
                tmp["feature_set"] = feature_set_name
                tmp["actual_tvt"] = actual
                tmp["pred_tvt"] = pred_tvt
                tmp["baseline_tvt"] = base
                tmp["residual"] = actual - pred_tvt
                tmp["baseline_residual"] = actual - base
                pred_parts.append(tmp)

    return pd.DataFrame(fold_rows), pd.concat(pred_parts, ignore_index=True), pd.concat(importance_parts, ignore_index=True) if importance_parts else pd.DataFrame()

## 6. Strict-CV Results

In [5]:
"Run strict-hybrid CV for legal and oracle feature families."

start = time.time()
strict_fold_metrics, strict_predictions, strict_importance = evaluate_fold_column(PRIMARY_FOLD, include_oracle=True)
print(f"strict_elapsed_s={time.time() - start:.1f}")

summary = (
    strict_fold_metrics.groupby(["label", "consumer", "feature_set"])
    .agg(
        rmse_mean=("rmse", "mean"),
        rmse_std=("rmse", "std"),
        mae_mean=("mae", "mean"),
        r2_mean=("r2", "mean"),
        last_known_hold_rmse_mean=("last_known_hold_rmse", "mean"),
        n_features=("n_features", "max"),
        n_valid_rows=("n_valid_rows", "sum"),
        best_iteration_mean=("best_iteration", "mean"),
    )
    .reset_index()
)
summary["rmse_delta_vs_hold"] = summary["rmse_mean"] - summary["last_known_hold_rmse_mean"]
summary["rmse_delta_vs_public_best"] = summary["rmse_mean"] - CONFIG["current_public_best_rmse"]
summary["rmse_delta_vs_known_state_local"] = summary["rmse_mean"] - CONFIG["best_xgb_known_state_local_rmse"]
summary = summary.sort_values("rmse_mean").reset_index(drop=True)
legal_summary = summary.loc[~summary["feature_set"].eq("oracle_direct_formations")].sort_values("rmse_mean").reset_index(drop=True)
best_legal = legal_summary.iloc[0].copy()
best_label = str(best_legal["label"])
best_feature_set = str(best_legal["feature_set"])
best_consumer = str(best_legal["consumer"])

print(summary.to_string(index=False))

C:\Users\sardo\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_huber.py:343: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)


C:\Users\sardo\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_huber.py:343: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)


C:\Users\sardo\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_huber.py:343: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)


C:\Users\sardo\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_huber.py:343: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)


C:\Users\sardo\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_huber.py:343: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)


C:\Users\sardo\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_huber.py:343: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)


strict_elapsed_s=548.8
                                                          label                           consumer                 feature_set  rmse_mean  rmse_std  mae_mean  r2_mean  last_known_hold_rmse_mean  n_features  n_valid_rows  best_iteration_mean  rmse_delta_vs_hold  rmse_delta_vs_public_best  rmse_delta_vs_known_state_local
   xgb_pseudohuber_known_state_recipe__oracle_direct_formations xgb_pseudohuber_known_state_recipe    oracle_direct_formations  12.879875  0.707148  9.279049 0.999561                  15.571277          63         92760                700.0           -2.691402                  -1.592125                        -0.553525
                          huber_delta__oracle_direct_formations                        huber_delta    oracle_direct_formations  13.131724  0.606685  9.567228 0.999545                  15.571277          63         92760                  NaN           -2.439553                  -1.340276                        -0.301676
              

## 7. Stress Diagnostics

In [6]:
"Run analog-distance folds for legal candidates and summarize stress slices."

stress_fold_metrics, stress_predictions, stress_importance = evaluate_fold_column(STRESS_FOLD, include_oracle=False)
stress_summary = (
    stress_fold_metrics.groupby(["label", "consumer", "feature_set"])
    .agg(
        stress_rmse_mean=("rmse", "mean"),
        stress_rmse_std=("rmse", "std"),
        stress_mae_mean=("mae", "mean"),
        stress_last_known_hold_rmse_mean=("last_known_hold_rmse", "mean"),
        stress_n_valid_rows=("n_valid_rows", "sum"),
    )
    .reset_index()
)
summary = summary.merge(stress_summary, on=["label", "consumer", "feature_set"], how="left")
summary["stress_minus_strict_rmse"] = summary["stress_rmse_mean"] - summary["rmse_mean"]
summary = summary.sort_values("rmse_mean").reset_index(drop=True)
legal_summary = summary.loc[~summary["feature_set"].eq("oracle_direct_formations")].sort_values("rmse_mean").reset_index(drop=True)
best_legal = legal_summary.iloc[0].copy()
best_label = str(best_legal["label"])

predictions = pd.concat([strict_predictions, stress_predictions], ignore_index=True)
predictions["progress_zone"] = pd.cut(predictions["eval_progress"], bins=[-0.001, 0.25, 0.50, 0.75, 1.001], labels=["z0", "z1", "z2", "z3"])
predictions["analog_distance_bin"] = pd.qcut(predictions["selected_analog_distance"], q=5, labels=["closest", "near", "middle", "far", "farthest"], duplicates="drop")

zone_summary = (
    predictions.groupby(["fold_column", "label", "progress_zone"], observed=True)
    .apply(lambda group: pd.Series({
        "n_rows": len(group),
        "rmse": rmse(group["actual_tvt"], group["pred_tvt"]),
        "baseline_rmse": rmse(group["actual_tvt"], group["baseline_tvt"]),
        "mean_residual": float(group["residual"].mean()),
        "mean_abs_residual": float(np.abs(group["residual"]).mean()),
        "mean_pseudo_nearest_abs_offset": float(group.filter(like="pseudo_nearest_abs_offset").iloc[:, 0].mean()),
        "mean_pseudo_top_margin": float(group.filter(like="pseudo_top_margin").iloc[:, 0].mean()),
        "mean_pseudo_std": float(group.filter(like="pseudo_mean_std").iloc[:, 0].mean()),
    }), include_groups=False)
    .reset_index()
)
zone_summary["rmse_delta_vs_hold"] = zone_summary["rmse"] - zone_summary["baseline_rmse"]

analog_bin_summary = (
    predictions.groupby(["fold_column", "label", "analog_distance_bin"], observed=True)
    .apply(lambda group: pd.Series({
        "n_rows": len(group),
        "n_wells": group["wellname"].nunique(),
        "mean_selected_analog_distance": float(group["selected_analog_distance"].mean()),
        "rmse": rmse(group["actual_tvt"], group["pred_tvt"]),
        "baseline_rmse": rmse(group["actual_tvt"], group["baseline_tvt"]),
    }), include_groups=False)
    .reset_index()
)
analog_bin_summary["rmse_delta_vs_hold"] = analog_bin_summary["rmse"] - analog_bin_summary["baseline_rmse"]

well_diagnostics = (
    predictions.loc[(predictions["fold_column"].eq(PRIMARY_FOLD)) & (predictions["label"].eq(best_label))]
    .groupby("wellname")
    .agg(
        n_rows=("actual_tvt", "size"),
        strict_fold=(PRIMARY_FOLD, "first"),
        analog_fold=(STRESS_FOLD, "first"),
        mean_selected_analog_distance=("selected_analog_distance", "mean"),
        mean_combined_stress=("combined_stress_score", "mean"),
        mean_residual=("residual", "mean"),
        mean_abs_residual=("residual", lambda series: float(np.abs(series).mean())),
        rmse=("residual", lambda series: float(np.sqrt(np.mean(np.square(series))))),
    )
    .reset_index()
    .sort_values("rmse", ascending=False)
)

importance = pd.concat([strict_importance, stress_importance], ignore_index=True)
importance_summary = (
    importance.groupby(["fold_column", "label", "consumer", "feature_set", "feature"])["importance"].mean().reset_index()
    .sort_values(["fold_column", "label", "importance"], ascending=[True, True, False])
)

print(summary.to_string(index=False))

C:\Users\sardo\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_huber.py:343: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)


C:\Users\sardo\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_huber.py:343: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)


C:\Users\sardo\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_huber.py:343: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)


C:\Users\sardo\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_huber.py:343: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)


                                                          label                           consumer                 feature_set  rmse_mean  rmse_std  mae_mean  r2_mean  last_known_hold_rmse_mean  n_features  n_valid_rows  best_iteration_mean  rmse_delta_vs_hold  rmse_delta_vs_public_best  rmse_delta_vs_known_state_local  stress_rmse_mean  stress_rmse_std  stress_mae_mean  stress_last_known_hold_rmse_mean  stress_n_valid_rows  stress_minus_strict_rmse
   xgb_pseudohuber_known_state_recipe__oracle_direct_formations xgb_pseudohuber_known_state_recipe    oracle_direct_formations  12.879875  0.707148  9.279049 0.999561                  15.571277          63         92760                700.0           -2.691402                  -1.592125                        -0.553525               NaN              NaN              NaN                               NaN                  NaN                       NaN
                          huber_delta__oracle_direct_formations                        huber

## 8. Artifact Export

In [7]:
"Export tables, run-bundle copies, model comparison, and a concise report."

best_legal_strict_rmse = float(best_legal["rmse_mean"])
best_legal_stress_rmse = float(best_legal["stress_rmse_mean"])
if best_legal_strict_rmse < CONFIG["best_xgb_known_state_local_rmse"] - CONFIG["promotion_margin"] and best_legal_stress_rmse <= CONFIG["best_xgb_known_state_stress_rmse"]:
    recommendation = "package_pseudo_top_xgboost"
elif best_legal_strict_rmse < CONFIG["current_public_best_rmse"]:
    recommendation = "iterate_before_package"
else:
    recommendation = "diagnostic_only"

findings = pd.DataFrame([
    {
        "finding": "best_legal_candidate",
        "detail": best_label,
        "value": best_legal_strict_rmse,
        "interpretation": f"Strict CV best legal RMSE with stress RMSE {best_legal_stress_rmse:.4f}.",
    },
    {
        "finding": "best_pseudo_top_candidate",
        "detail": str(legal_summary.loc[legal_summary["feature_set"].str.contains("pseudo_top", regex=False)].iloc[0]["label"]),
        "value": float(legal_summary.loc[legal_summary["feature_set"].str.contains("pseudo_top", regex=False)].iloc[0]["rmse_mean"]),
        "interpretation": "Best candidate using pseudo-top features.",
    },
    {
        "finding": "nearest_top_proxy_accuracy",
        "detail": PRIMARY_FOLD,
        "value": float(data[f"pseudo_nearest_matches_label__{PRIMARY_FOLD}"].mean()),
        "interpretation": "Agreement between fold-safe nearest pseudo top and direct nearest formation label.",
    },
    {
        "finding": "decision",
        "detail": recommendation,
        "value": best_legal_strict_rmse,
        "interpretation": "Promotion rule based on strict CV, analog-distance stress, and current public/local baselines.",
    },
])

paths = {
    "summary": TABLE_DIR / "model__pseudo_top_uncertainty_summary__2026-06-30.csv",
    "fold_metrics": TABLE_DIR / "model__pseudo_top_uncertainty_fold_metrics__2026-06-30.csv",
    "zone_summary": TABLE_DIR / "model__pseudo_top_uncertainty_zone_summary__2026-06-30.csv",
    "analog_bin_summary": TABLE_DIR / "model__pseudo_top_uncertainty_analog_bin_summary__2026-06-30.csv",
    "well_diagnostics": TABLE_DIR / "model__pseudo_top_uncertainty_well_diagnostics__2026-06-30.csv",
    "feature_importance": TABLE_DIR / "model__pseudo_top_uncertainty_feature_importance__2026-06-30.csv",
    "findings": TABLE_DIR / "model__pseudo_top_uncertainty_findings__2026-06-30.csv",
}
exports = {
    "summary": summary,
    "fold_metrics": pd.concat([strict_fold_metrics, stress_fold_metrics], ignore_index=True),
    "zone_summary": zone_summary,
    "analog_bin_summary": analog_bin_summary,
    "well_diagnostics": well_diagnostics,
    "feature_importance": importance_summary,
    "findings": findings,
}
for key, frame in exports.items():
    frame.to_csv(paths[key], index=False)
    shutil.copy2(paths[key], RUN_DIR / paths[key].name)

comparison_path = TABLE_DIR / "model_comparison.csv"
comparison = pd.read_csv(comparison_path) if comparison_path.exists() else pd.DataFrame()
new_comparison = pd.DataFrame([{
    "run_id": RUN_ID,
    "model_family": "alignment",
    "model_name": "pseudo_top_uncertainty",
    "best_candidate": best_label,
    "local_cv_rmse": best_legal_strict_rmse,
    "local_cv_mae": float(best_legal["mae_mean"]),
    "public_lb_rmse": np.nan,
    "status": recommendation,
    "notes": f"stress_rmse={best_legal_stress_rmse:.4f}; nearest_top_accuracy={data[f'pseudo_nearest_matches_label__{PRIMARY_FOLD}'].mean():.4f}",
    "updated_at": RUN_DATE,
}])
if not comparison.empty and "run_id" in comparison.columns:
    comparison = comparison.loc[comparison["run_id"] != RUN_ID].copy()
comparison = pd.concat([comparison, new_comparison], ignore_index=True, sort=False)
comparison.to_csv(comparison_path, index=False)
shutil.copy2(comparison_path, RUN_DIR / comparison_path.name)

report_path = REPORT_DIR / "model__pseudo_top_uncertainty__2026-06-30.md"
best_pseudo = legal_summary.loc[legal_summary["feature_set"].str.contains("pseudo_top", regex=False)].iloc[0]
report = f"""# Pseudo-Top Uncertainty Feature Screen

## Result

Best legal strict-CV candidate: `{best_label}` with RMSE {best_legal_strict_rmse:.4f}, MAE {float(best_legal["mae_mean"]):.4f}, and analog-distance stress RMSE {best_legal_stress_rmse:.4f}.

Best pseudo-top feature candidate: `{best_pseudo["label"]}` with strict-CV RMSE {float(best_pseudo["rmse_mean"]):.4f}.

## Pseudo-Top Quality

Nearest pseudo-top agreement with direct nearest formation labels was {data[f"pseudo_nearest_matches_label__{PRIMARY_FOLD}"].mean():.4f} under `{PRIMARY_FOLD}` and {data[f"pseudo_nearest_matches_label__{STRESS_FOLD}"].mean():.4f} under `{STRESS_FOLD}`.

Mean top-elevation RMSE across formation proxies was {proxy_quality.loc[proxy_quality["formation"].isin(FORMATION_COLS), "top_rmse"].mean():.4f}.

## Downstream RMSE

Pseudo-top features did not use direct formation columns. Oracle diagnostics remain separate and should only be read as a recoverable-signal ceiling.

## Leakage Notes

Formation top columns were used only as fold-local labels and oracle diagnostics. For each validation fold, pseudo-top models were trained on the complementary wells and predicted held-out rows. The exported legal pseudo features are therefore hidden-test-compatible if the same fitting recipe is rerun inside a Kaggle notebook.

## Decision

Recommendation: `{recommendation}`.

## Next Move

Use this screen to decide whether to integrate pseudo-top uncertainty into the gated Bayesian regime, because the gate can use margin and uncertainty as shrinkage controls rather than raw predictors.
"""
report_path.write_text(report, encoding="utf-8")
shutil.copy2(report_path, RUN_DIR / report_path.name)

print(findings.to_string(index=False))
print(report_path)

                   finding                                                          detail     value                                                                                 interpretation
      best_legal_candidate                           xgb_square_regularized__safe_typewell 13.450635                                            Strict CV best legal RMSE with stress RMSE 13.8870.
 best_pseudo_top_candidate xgb_pseudohuber_known_state_recipe__safe_pseudo_top_uncertainty 13.589132                                                      Best candidate using pseudo-top features.
nearest_top_proxy_accuracy                                              strict_hybrid_fold  0.580433             Agreement between fold-safe nearest pseudo top and direct nearest formation label.
                  decision                                          iterate_before_package 13.450635 Promotion rule based on strict CV, analog-distance stress, and current public/local baselines.
C:\Users\sardo\OneDr

## 9. Findings

In [8]:
"Show the headline artifacts for notebook readers."

display(summary.head(12))
display(proxy_quality.groupby(["fold_column", "formation"], dropna=False).agg(top_rmse=("top_rmse", "mean"), top_mae=("top_mae", "mean"), mean_ensemble_std=("mean_ensemble_std", "mean"), nearest_accuracy=("nearest_accuracy", "mean")).reset_index())
display(findings)

,label,consumer,feature_set,rmse_mean,rmse_std,mae_mean,r2_mean,last_known_hold_rmse_mean,n_features,n_valid_rows,best_iteration_mean,rmse_delta_vs_hold,rmse_delta_vs_public_best,rmse_delta_vs_known_state_local,stress_rmse_mean,stress_rmse_std,stress_mae_mean,stress_last_known_hold_rmse_mean,stress_n_valid_rows,stress_minus_strict_rmse
0,xgb_pseudohuber_known_state_recipe__oracle_dir...,xgb_pseudohuber_known_state_recipe,oracle_direct_formations,12.879875,0.707148,9.279049,0.999561,15.571277,63,92760,700.0,-2.691402,-1.592125,-0.553525,NaN,NaN,NaN,NaN,NaN,NaN
1,huber_delta__oracle_direct_formations,huber_delta,oracle_direct_formations,13.131724,0.606685,9.567228,0.999545,15.571277,63,92760,NaN,-2.439553,-1.340276,-0.301676,NaN,NaN,NaN,NaN,NaN,NaN
2,xgb_square_regularized__safe_typewell,xgb_square_regularized,safe_typewell,13.450635,0.394635,9.527273,0.999523,15.571277,41,92760,550.0,-2.120642,-1.021365,0.017235,13.886984,0.548526,10.080152,15.581239,92760.0,0.436349
3,xgb_pseudohuber_known_state_recipe__safe_typewell,xgb_pseudohuber_known_state_recipe,safe_typewell,13.469342,0.872132,9.510546,0.999519,15.571277,41,92760,700.0,-2.101935,-1.002658,0.035942,13.854245,0.681569,9.886458,15.581239,92760.0,0.384903
4,xgb_pseudohuber_known_state_recipe__safe_base,xgb_pseudohuber_known_state_recipe,safe_base,13.484648,0.934388,9.496713,0.999517,15.571277,36,92760,700.0,-2.086629,-0.987352,0.051248,13.820020,0.667947,9.869065,15.581239,92760.0,0.335372
5,xgb_square_regularized__safe_base,xgb_square_regularized,safe_base,13.487823,0.360565,9.521851,0.999521,15.571277,36,92760,550.0,-2.083454,-0.984177,0.054423,13.850704,0.545147,10.051667,15.581239,92760.0,0.362881
6,xgb_square_regularized__safe_typewell_state,xgb_square_regularized,safe_typewell_state,13.517257,0.470571,9.572704,0.999518,15.571277,49,92760,550.0,-2.054020,-0.954743,0.083857,13.912686,0.514319,10.080527,15.581239,92760.0,0.395430
7,xgb_pseudohuber_known_state_recipe__safe_typew...,xgb_pseudohuber_known_state_recipe,safe_typewell_state,13.532542,0.925455,9.544312,0.999514,15.571277,49,92760,700.0,-2.038735,-0.939458,0.099142,13.913177,0.718841,9.966707,15.581239,92760.0,0.380635
8,xgb_pseudohuber_known_state_recipe__safe_pseud...,xgb_pseudohuber_known_state_recipe,safe_pseudo_top_uncertainty,13.589132,0.894962,9.589422,0.999510,15.571277,71,92760,700.0,-1.982145,-0.882868,0.155732,13.995012,0.596060,9.990585,15.581239,92760.0,0.405880
9,xgb_pseudohuber_known_state_recipe__safe_pseud...,xgb_pseudohuber_known_state_recipe,safe_pseudo_top,13.594596,0.961219,9.583724,0.999509,15.571277,56,92760,700.0,-1.976681,-0.877404,0.161196,13.983006,0.536852,9.969652,15.581239,92760.0,0.388411


,fold_column,formation,top_rmse,top_mae,mean_ensemble_std,nearest_accuracy
0,analog_distance_fold,ANCC,36.115667,27.307085,2.252600,NaN
1,analog_distance_fold,ASTNL,36.141518,26.902796,2.586729,NaN
2,analog_distance_fold,ASTNU,37.748549,28.102448,2.489752,NaN
3,analog_distance_fold,BUDA,34.471779,25.013579,2.370127,NaN
4,analog_distance_fold,EGFDL,34.509167,25.367227,2.504926,NaN
5,analog_distance_fold,EGFDU,34.209112,25.063224,2.354379,NaN
6,analog_distance_fold,nearest_classifier,NaN,NaN,2.426611,0.582277
7,strict_hybrid_fold,ANCC,35.208638,26.607742,2.021169,NaN
8,strict_hybrid_fold,ASTNL,35.230138,26.536828,2.328327,NaN
9,strict_hybrid_fold,ASTNU,36.706530,27.494402,2.655390,NaN


,finding,detail,value,interpretation
0,best_legal_candidate,xgb_square_regularized__safe_typewell,13.450635,Strict CV best legal RMSE with stress RMSE 13....
1,best_pseudo_top_candidate,xgb_pseudohuber_known_state_recipe__safe_pseud...,13.589132,Best candidate using pseudo-top features.
2,nearest_top_proxy_accuracy,strict_hybrid_fold,0.580433,Agreement between fold-safe nearest pseudo top...
3,decision,iterate_before_package,13.450635,"Promotion rule based on strict CV, analog-dist..."
